In [20]:
import pandas as pd
import numpy as np
df = pd.read_parquet(
    "../Data/Processed/electricity_load.parquet"
)

In [ ]:
# Inspect the raw dataset structure
print("Dataset shape:", df.shape)

print("\nFirst five rows:")
display(df.head())

print("\nIndex information:")
print(df.index)

print("\nColumn count:", len(df.columns))
print("Data types:")
print(df.dtypes.value_counts())

In [3]:
# Aggregate electricity consumption across all 370 clients
total_demand = df.sum(axis=1)

# Give the resulting Series a meaningful name
total_demand.name = "total_demand"

# Inspect the result
print("Shape:", total_demand.shape)
print("\nFirst 10 observations:")
display(total_demand.head(10))

print("\nLast 10 observations:")
display(total_demand.tail(10))

Shape: (140256,)

First 10 observations:


2011-01-01 00:15:00    68513.115338
2011-01-01 00:30:00    69180.304361
2011-01-01 00:45:00    69364.850573
2011-01-01 01:00:00    68350.480660
2011-01-01 01:15:00    66166.874303
2011-01-01 01:30:00    64843.191940
2011-01-01 01:45:00    66017.963844
2011-01-01 02:00:00    65345.513794
2011-01-01 02:15:00    65567.578171
2011-01-01 02:30:00    65765.802247
Name: total_demand, dtype: float64


Last 10 observations:


2014-12-31 21:45:00    153905.565556
2014-12-31 22:00:00    150568.543355
2014-12-31 22:15:00    148365.634998
2014-12-31 22:30:00    144894.923782
2014-12-31 22:45:00    137023.682306
2014-12-31 23:00:00    132288.754939
2014-12-31 23:15:00    128543.516567
2014-12-31 23:30:00    127478.208411
2014-12-31 23:45:00    125641.710272
2015-01-01 00:00:00    124214.199215
Name: total_demand, dtype: float64

In [4]:
print("Number of zero-demand observations:", (total_demand == 0).sum())
print("Minimum demand:", total_demand.min())
print("Maximum demand:", total_demand.max())
print("Mean demand:", total_demand.mean())

Number of zero-demand observations: 0
Minimum demand: 342.5955085040956
Maximum demand: 452652.5955674096
Mean demand: 195556.9425263531


In [7]:
# Resample the 15-minute total demand into hourly average demand
hourly_demand = total_demand.resample("h").mean()

# Give the series a clear name
hourly_demand.name = "total_demand"

# Inspect the result
print("Shape:", hourly_demand.shape)

print("\nFirst 10 observations:")
display(hourly_demand.head(10))

print("\nLast 10 observations:")
display(hourly_demand.tail(10))

Shape: (35065,)

First 10 observations:


2011-01-01 00:00:00    69019.423424
2011-01-01 01:00:00    66344.627687
2011-01-01 02:00:00    65981.054883
2011-01-01 03:00:00    66576.533566
2011-01-01 04:00:00    64963.552675
2011-01-01 05:00:00    65183.209978
2011-01-01 06:00:00    68489.571302
2011-01-01 07:00:00    70503.796317
2011-01-01 08:00:00    71888.836031
2011-01-01 09:00:00    73955.676155
Freq: h, Name: total_demand, dtype: float64


Last 10 observations:


2014-12-31 15:00:00    237544.997937
2014-12-31 16:00:00    232367.180784
2014-12-31 17:00:00    235987.577737
2014-12-31 18:00:00    236020.817245
2014-12-31 19:00:00    202877.313799
2014-12-31 20:00:00    166951.192473
2014-12-31 21:00:00    156247.443652
2014-12-31 22:00:00    145213.196110
2014-12-31 23:00:00    128488.047547
2015-01-01 00:00:00    124214.199215
Freq: h, Name: total_demand, dtype: float64

In [8]:
print("Missing hourly values:", hourly_demand.isna().sum())
print("Number of hourly observations:", len(hourly_demand))

print("\nTime range:")
print(hourly_demand.index.min(), "to", hourly_demand.index.max())

print("\nTime differences:")
print(hourly_demand.index.to_series().diff().value_counts().head())

Missing hourly values: 0
Number of hourly observations: 35065

Time range:
2011-01-01 00:00:00 to 2015-01-01 00:00:00

Time differences:
0 days 01:00:00    35064
Name: count, dtype: int64


In [9]:
# Create the base modelling DataFrame
data = hourly_demand.to_frame()

# Inspect it
print("Shape:", data.shape)
display(data.head())

Shape: (35065, 1)


,total_demand
2011-01-01 00:00:00,69019.423424
2011-01-01 01:00:00,66344.627687
2011-01-01 02:00:00,65981.054883
2011-01-01 03:00:00,66576.533566
2011-01-01 04:00:00,64963.552675


In [11]:
# Create the one-hour-ahead forecasting target
data["target"] = data["total_demand"].shift(-1)

# Inspect the relationship
display(data.head(10))

,total_demand,target
2011-01-01 00:00:00,69019.423424,66344.627687
2011-01-01 01:00:00,66344.627687,65981.054883
2011-01-01 02:00:00,65981.054883,66576.533566
2011-01-01 03:00:00,66576.533566,64963.552675
2011-01-01 04:00:00,64963.552675,65183.209978
2011-01-01 05:00:00,65183.209978,68489.571302
2011-01-01 06:00:00,68489.571302,70503.796317
2011-01-01 07:00:00,70503.796317,71888.836031
2011-01-01 08:00:00,71888.836031,73955.676155
2011-01-01 09:00:00,73955.676155,73634.809216


In [12]:
# Verify that target(t) equals demand(t+1)
print("Demand at 01:00:", data.loc["2011-01-01 01:00", "total_demand"])
print("Target at 00:00:", data.loc["2011-01-01 00:00", "target"])

Demand at 01:00: 66344.62768663246
Target at 00:00: 66344.62768663246


In [13]:
data["lag_1"] = data["total_demand"].shift(1)

In [14]:
display(data.head(5))

,total_demand,target,lag_1
2011-01-01 00:00:00,69019.423424,66344.627687,NaN
2011-01-01 01:00:00,66344.627687,65981.054883,69019.423424
2011-01-01 02:00:00,65981.054883,66576.533566,66344.627687
2011-01-01 03:00:00,66576.533566,64963.552675,65981.054883
2011-01-01 04:00:00,64963.552675,65183.209978,66576.533566


In [15]:
print(
    "Demand at 00:00:",
    data.loc["2011-01-01 00:00", "total_demand"]
)

print(
    "Lag_1 at 01:00:",
    data.loc["2011-01-01 01:00", "lag_1"]
)

Demand at 00:00: 69019.42342410504
Lag_1 at 01:00: 69019.42342410504


In [16]:
print(
    "Target at 00:00:",
    data.loc["2011-01-01 00:00", "target"]
)

print(
    "Demand at 01:00:",
    data.loc["2011-01-01 01:00", "total_demand"]
)

Target at 00:00: 66344.62768663246
Demand at 01:00: 66344.62768663246


In [17]:
# Create historical demand lag features
lag_hours = [2, 3, 24, 48, 168]

for lag in lag_hours:
    data[f"lag_{lag}"] = data["total_demand"].shift(lag)

# Inspect the resulting columns
print(data.columns.tolist())

['total_demand', 'target', 'lag_1', 'lag_2', 'lag_3', 'lag_24', 'lag_48', 'lag_168']


In [ ]:
display(data.head(10))

In [19]:
current_time = "2011-01-08 00:00:00"
previous_week = "2011-01-01 00:00:00"

print(
    "Demand one week earlier:",
    data.loc[previous_week, "total_demand"]
)

print(
    "lag_168 at current time:",
    data.loc[current_time, "lag_168"]
)

Demand one week earlier: 69019.42342410504
lag_168 at current time: 69019.42342410504


In [21]:
# Calendar features
data["hour"] = data.index.hour
data["day_of_week"] = data.index.dayofweek
data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)

# Cyclical encoding for hour of day
data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

# Inspect
display(data[[
    "hour",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos"
]].head(30))

,hour,day_of_week,is_weekend,hour_sin,hour_cos
2011-01-01 00:00:00,0,5,1,0.000000e+00,1.000000e+00
2011-01-01 01:00:00,1,5,1,2.588190e-01,9.659258e-01
2011-01-01 02:00:00,2,5,1,5.000000e-01,8.660254e-01
2011-01-01 03:00:00,3,5,1,7.071068e-01,7.071068e-01
2011-01-01 04:00:00,4,5,1,8.660254e-01,5.000000e-01
2011-01-01 05:00:00,5,5,1,9.659258e-01,2.588190e-01
2011-01-01 06:00:00,6,5,1,1.000000e+00,6.123234e-17
2011-01-01 07:00:00,7,5,1,9.659258e-01,-2.588190e-01
2011-01-01 08:00:00,8,5,1,8.660254e-01,-5.000000e-01
2011-01-01 09:00:00,9,5,1,7.071068e-01,-7.071068e-01


In [22]:
display(
    data.loc[
        data["hour"].isin([0, 1, 23]),
        ["hour", "hour_sin", "hour_cos"]
    ].head(10)
)

,hour,hour_sin,hour_cos
2011-01-01 00:00:00,0,0.000000,1.000000
2011-01-01 01:00:00,1,0.258819,0.965926
2011-01-01 23:00:00,23,-0.258819,0.965926
2011-01-02 00:00:00,0,0.000000,1.000000
2011-01-02 01:00:00,1,0.258819,0.965926
2011-01-02 23:00:00,23,-0.258819,0.965926
2011-01-03 00:00:00,0,0.000000,1.000000
2011-01-03 01:00:00,1,0.258819,0.965926
2011-01-03 23:00:00,23,-0.258819,0.965926
2011-01-04 00:00:00,0,0.000000,1.000000


In [23]:
data["rolling_mean_24"] = data["total_demand"].rolling(24).mean()
data["rolling_std_24"] = data["total_demand"].rolling(24).std()

In [24]:
display(
    data[
        [
            "total_demand",
            "lag_1",
            "lag_24",
            "rolling_mean_24",
            "rolling_std_24"
        ]
    ].iloc[20:30]
)

,total_demand,lag_1,lag_24,rolling_mean_24,rolling_std_24
2011-01-01 20:00:00,78806.030818,79348.818016,NaN,NaN,NaN
2011-01-01 21:00:00,77966.962289,78806.030818,NaN,NaN,NaN
2011-01-01 22:00:00,74463.373943,77966.962289,NaN,NaN,NaN
2011-01-01 23:00:00,70324.672302,74463.373943,NaN,72104.124008,4415.746233
2011-01-02 00:00:00,68905.097362,70324.672302,69019.423424,72099.360422,4419.278858
2011-01-02 01:00:00,65707.994644,68905.097362,66344.627687,72072.834045,4457.072102
2011-01-02 02:00:00,65186.796037,65707.994644,65981.054883,72039.739927,4506.940398
2011-01-02 03:00:00,65238.948686,65186.796037,66576.533566,71984.007224,4585.029329
2011-01-02 04:00:00,66189.772239,65238.948686,64963.552675,72035.099705,4509.608521
2011-01-02 05:00:00,74121.693906,66189.772239,65183.209978,72407.536536,4282.510180


In [25]:
print("Shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

print("\nMissing values:")
display(data.isna().sum())

Shape: (35065, 15)

Columns:
['total_demand', 'target', 'lag_1', 'lag_2', 'lag_3', 'lag_24', 'lag_48', 'lag_168', 'hour', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'rolling_mean_24', 'rolling_std_24']

Missing values:


total_demand         0
target               1
lag_1                1
lag_2                2
lag_3                3
lag_24              24
lag_48              48
lag_168            168
hour                 0
day_of_week          0
is_weekend           0
hour_sin             0
hour_cos             0
rolling_mean_24     23
rolling_std_24      23
dtype: int64

In [27]:
data = data.dropna().copy()

In [28]:
print("Shape after dropping NaNs:", data.shape)
print("\nMissing values:")
print(data.isna().sum().sum())

print("\nFirst timestamp:", data.index.min())
print("Last timestamp:", data.index.max())

Shape after dropping NaNs: (34896, 15)

Missing values:
0

First timestamp: 2011-01-08 00:00:00
Last timestamp: 2014-12-31 23:00:00


In [29]:
n = len(data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

print("Total observations:", n)
print("Training observations:", train_end)
print("Validation observations:", val_end - train_end)
print("Test observations:", n - val_end)

print("\nTraining ends:", data.index[train_end - 1])
print("Validation ends:", data.index[val_end - 1])
print("Test starts:", data.index[val_end])
print("Test ends:", data.index[-1])

Total observations: 34896
Training observations: 24427
Validation observations: 5234
Test observations: 5235

Training ends: 2013-10-21 18:00:00
Validation ends: 2014-05-27 20:00:00
Test starts: 2014-05-27 21:00:00
Test ends: 2014-12-31 23:00:00


In [30]:
feature_cols = [col for col in data.columns if col != "target"]

X = data[feature_cols]
y = data["target"]

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

In [31]:
print("X shapes:")
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("\ny shapes:")
print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

print("\nDate ranges:")
print("Train:", X_train.index.min(), "→", X_train.index.max())
print("Val:  ", X_val.index.min(), "→", X_val.index.max())
print("Test: ", X_test.index.min(), "→", X_test.index.max())

X shapes:
X_train: (24427, 14)
X_val:   (5234, 14)
X_test:  (5235, 14)

y shapes:
y_train: (24427,)
y_val:   (5234,)
y_test:  (5235,)

Date ranges:
Train: 2011-01-08 00:00:00 → 2013-10-21 18:00:00
Val:   2013-10-21 19:00:00 → 2014-05-27 20:00:00
Test:  2014-05-27 21:00:00 → 2014-12-31 23:00:00


In [34]:
print(X_train.columns.tolist())

['total_demand', 'lag_1', 'lag_2', 'lag_3', 'lag_24', 'lag_48', 'lag_168', 'hour', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'rolling_mean_24', 'rolling_std_24']


In [33]:
display(X_train.head())

,total_demand,lag_1,lag_2,lag_3,lag_24,lag_48,lag_168,hour,day_of_week,is_weekend,hour_sin,hour_cos,rolling_mean_24,rolling_std_24
2011-01-08 00:00:00,86403.301624,111609.594412,130176.983542,141034.492592,85716.807571,85490.070525,69019.423424,0,5,1,0.000000,1.000000,120015.695552,29768.012259
2011-01-08 01:00:00,72534.984171,86403.301624,111609.594412,130176.983542,73297.911807,72602.273897,66344.627687,1,5,1,0.258819,0.965926,119983.906900,29820.431514
2011-01-08 02:00:00,69325.030607,72534.984171,86403.301624,111609.594412,70543.526944,69897.630958,65981.054883,2,5,1,0.500000,0.866025,119933.136219,29909.171077
2011-01-08 03:00:00,68554.184257,69325.030607,72534.984171,86403.301624,70242.037536,69467.257186,66576.533566,3,5,1,0.707107,0.707107,119862.809000,30032.821574
2011-01-08 04:00:00,69033.522311,68554.184257,69325.030607,72534.984171,68884.017033,69493.357908,64963.552675,4,5,1,0.866025,0.500000,119869.038386,30021.801334


In [35]:
from pathlib import Path

processed_dir = Path("../Data/Processed")
processed_dir.mkdir(parents=True, exist_ok=True)

feature_path = processed_dir / "hourly_demand_features.csv"

data.to_csv(feature_path, index=True)

print(f"Saved feature-engineered data to: {feature_path}")

Saved feature-engineered data to: ..\Data\Processed\hourly_demand_features.csv


In [36]:
print("File exists:", feature_path.exists())
print("File size:", feature_path.stat().st_size, "bytes")

File exists: True
File size: 8712485 bytes


In [37]:
saved_data = pd.read_csv(
    feature_path,
    index_col=0,
    parse_dates=True
)

print("Saved shape:", saved_data.shape)
print("Loaded index type:", type(saved_data.index))
print("Missing values:", saved_data.isna().sum().sum())

Saved shape: (34896, 15)
Loaded index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Missing values: 0
